# Station charges — 01 source extraction

Single source of truth for the station charge source register: what an
infrastructure manager or station operator charges for a passenger train
calling at a station. Every row is written by this notebook;
`data/sources_register.csv` is a generated artifact and must never be
hand-edited.

Run this before `02_station_charges.ipynb`, which fails rather than citing a
source that does not resolve against this register.

Same contract as `tac/calib/`, `energy_pricing/calib/` and `facility/calib/`:
notebook is truth, CSV is output. Put the documents themselves in `sources/`
(gitignored — they are publishers' PDFs and spreadsheets, not ours to
redistribute) and register them here by filename.

In [1]:
# Station charges — source extraction
#
# Station charge tariffs come from three kinds of document, and each is read
# differently in 02: a station price list (usually PDF), a network statement
# annex (PDF or XLSX), and figures transcribed by hand from a source that
# cannot be parsed at all. The register does not care which — it records what
# the document is, so a value can always be traced back to it.

import csv
from pathlib import Path


def _resolve_data_dir() -> Path:
    """Notebook may run from charges/ or from the repo root; resolve either."""
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/stops/charges/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the charges data directory from {here}")


DATA_DIR = _resolve_data_dir()
SOURCES_DIR = DATA_DIR.parent / "sources"
SOURCES_DIR.mkdir(exist_ok=True)
print(f"data directory:    {DATA_DIR}")
print(f"source documents:  {SOURCES_DIR}")

# Date the register was last reviewed end to end.
REGISTER_REVIEWED = "TO_VERIFY"

REGISTER_COLUMNS = [
    "source_id",
    "short_id",
    "used",
    "downloaded",
    "title",
    "publisher",
    "pub_year",
    "price_basis_year",
    "currency",
    "kind",
    "url_or_file",
    "date_accessed",
    "reliability_note",
]

register_rows: list[tuple] = []

data directory:    C:\Users\david\PycharmProjects\night-train-target-network\backend\models\infrastructure\stops\charges\data
source documents:  C:\Users\david\PycharmProjects\night-train-target-network\backend\models\infrastructure\stops\charges\sources


## Station price lists and network statement annexes

One row per document. `kind` drives nothing mechanically but tells the reader
in `02` which extraction path a document needs:

| kind | typical format | how `02` reads it |
|---|---|---|
| `station_price_list` | PDF | `pdf_table` if the tables are machine-readable, otherwise `manual` |
| `network_statement` | PDF or XLSX | `xlsx_table` for annexes published as spreadsheets |
| `operator_model` | XLSX | `xlsx_table` |
| `manual_transcription` | anything | `manual` — figures typed into the notebook, with the page cited |

`used` is `Used` once a value in `02` cites it, `Registered` while it is only
listed. `downloaded` is `x` when the file is in `sources/`.

**Josh:** add your sources here first, then read them in `02`. Leave
`price_basis_year` as the year the tariff applies to, not the year of
publication — the two differ in most network statements.

In [2]:
# --- Station charge sources ---
_R = REGISTER_REVIEWED

register_rows += [
    (
        "DE-DB-SPL-2026",
        "de_db_spl_2026",
        "Registered",
        "",
        "Stationspreisliste 2026",
        "DB InfraGO AG",
        2025,
        2026,
        "EUR",
        "station_price_list",
        "de_db_stationspreisliste_2026.pdf",
        _R,
        "Per-call charge by station category and Land. TO_VERIFY: which "
        "category each catalog stop falls into.",
    ),
    # Josh: further sources go here. One tuple per document, same column order
    # as REGISTER_COLUMNS. Keep the id stable once a value in 02 cites it.
]

## The values carried over from the retired curated catalog

Thirteen stations carried a `stop_charge_eur` in `db/dev/seed.py`'s curated
list, which the stop classification pipeline replaced. `seed.py` attributed
them to "Illustrative / internal estimate" — they are **not** published
tariffs, and every one should be replaced by a sourced figure. Registered here
so the values are traceable rather than silently inherited.

In [3]:
register_rows += [
    (
        "ILLUSTRATIVE-CURATED",
        "illustrative_curated",
        "Used",
        "",
        "Curated stop catalog placeholder charges (retired 2026-08-18)",
        "Back-on-Track (internal)",
        2026,
        2026,
        "EUR",
        "manual_transcription",
        "db/dev/seed.py, _STOP_INFRASTRUCTURES_CANONICAL before its removal",
        _R,
        "NOT a published tariff. Illustrative internal estimates, kept only so "
        "the figures are not lost. Replace each one with a sourced value.",
    ),
]

## Write and validate

In [4]:
def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    """STDLIB-ONLY writer, shared by both charge notebooks."""
    path = DATA_DIR / name
    with open(path, "w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    print(f"  {name}: {len(rows)} rows")


rows = [dict(zip(REGISTER_COLUMNS, row, strict=True)) for row in register_rows]

ids = [row["source_id"] for row in rows]
duplicates = sorted({i for i in ids if ids.count(i) > 1})
if duplicates:
    raise ValueError(f"duplicate source_id: {duplicates}")

missing_files = [
    row["url_or_file"]
    for row in rows
    if row["downloaded"] == "x" and not (SOURCES_DIR / row["url_or_file"]).is_file()
]
if missing_files:
    print(f"\n  WARNING: marked downloaded but absent from sources/: {missing_files}")

write_data("sources_register.csv", REGISTER_COLUMNS, rows)
print(
    f"register: {len(rows)} sources, "
    f"{sum(1 for r in rows if r['used'] == 'Used')} in use, "
    f"{sum(1 for r in rows if r['downloaded'] == 'x')} documents on disk"
)

  sources_register.csv: 2 rows
register: 2 sources, 1 in use, 0 documents on disk
